# Day 2 (teammate A) - ASL words. CPU is enough; no Day-1 data needed.
Inputs: `slr-code`, `209sontung/sign-language`, competition `asl-signs`. Internet ON.

In [ ]:
# Finds the uploaded "slr-code" dataset wherever Kaggle mounts it (the path differs between Kaggle versions).
import glob, os, shutil, subprocess, sys, zipfile
ON_KAGGLE = os.path.exists("/kaggle/input")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
CODE = f"{WORK}/code"
hits = [h for d in ("*", "*/*", "*/*/*") for h in glob.glob(f"/kaggle/input/{d}/slr/__init__.py")] + glob.glob("/content/slr_code/**/slr/__init__.py", recursive=True)  # shallow search: ASL Alphabet has ~87k files
if hits:
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE, dirs_exist_ok=True)
else:
    zips = [h for d in ("*", "*/*", "*/*/*") for h in glob.glob(f"/kaggle/input/{d}/slr_code.zip")] + glob.glob("/content/**/slr_code.zip", recursive=True)
    if zips:
        zipfile.ZipFile(zips[0]).extractall(CODE)
    else:
        print("slr-code NOT FOUND. Add it: right sidebar -> + Add Input -> Your Datasets -> slr-code. Currently attached:")
        for p in sorted(glob.glob("/kaggle/input/*") + glob.glob("/kaggle/input/*/*")):
            print("  ", p)
        raise SystemExit("Attach the slr-code dataset, then run this cell again.")
os.chdir(CODE)
print("code ready in", CODE, os.listdir(CODE))
# mediapipe 0.10.14 still has the `mp.solutions` API we use. If pip complains about numpy, run
# `!pip install "numpy<2"` and restart the session once.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mediapipe==0.10.14", "remotezip", "opencv-python-headless"])

## 1. Pretrained inference on the chosen signs (no training)
Set `MODEL_DIR` to the pretrained-model folder shown in the sidebar (hover -> copy path).

In [ ]:
MODEL_DIR = "/kaggle/input/sign-language"
SIGNS = "hello thankyou please yes no mom dad water eat drink go sleep happy sad like want home cat dog bird red blue green hungry sick".split()
!python scripts/asl_words.py inspect --model-dir $MODEL_DIR
!python scripts/asl_words.py infer --model-dir $MODEL_DIR --data /kaggle/input/asl-signs --signs {" ".join(SIGNS)} --per-sign 20 --out $WORK/models/asl_pretrained_eval.json

## 2. Train only on those signs (run this if step 1 failed, or if the team lead asks)
Also fine to run even if step 1 worked: we then compare both.

In [ ]:
!python scripts/asl_words.py extract --data /kaggle/input/asl-signs --signs {" ".join(SIGNS)} --out $WORK/asl_words
!python scripts/train_words.py --data $WORK/asl_words --out $WORK/models/asl_words.pt --epochs 120

## 3. Package for download
Also copy the pretrained model files so the demo can use them.

In [ ]:
!cp $MODEL_DIR/*.tflite $MODEL_DIR/*.json $WORK/models/ 2>/dev/null
!cd $WORK && zip -r asl_models.zip models && ls -lh asl_models.zip

Then **Save Version -> Save & Run All**, open the Output tab and download `asl_models.zip`.